In [ ]:
# Colab setup: clone data assets when this notebook runs in Google Colab
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab] Cloning repository assets...")
    !git clone https://github.com/mattjunior039/CampusAIAssistantTutorial.git _repo_tmp
    !cp -r _repo_tmp/data . 2>/dev/null || true
    !rm -rf _repo_tmp
    print("[Colab] Ready.")

# Project 1: AI Campus Assistant Pipeline
## Phase 2: Dense Semantic Embeddings and Intent Classification

This notebook moves from exact word matching to semantic matching. Instead of asking whether two questions share the same words, we ask whether they live in the same neighborhood of meaning.

By the end, you will inspect raw 384-dimensional embeddings, compare distances between sentence vectors, train an intent classifier, visualize semantic neighborhoods in 2D and 3D, and design a fallback for questions outside the known campus categories.

## 1. From word overlap to a meaning map

In Phase 1, a question like `Where can I park my car?` could fail against a policy written with terms like `commuter vehicle permit`. The intent is similar, but the words are different.

Dense embeddings solve this by placing sentences onto a meaning map. Nearby points usually express related ideas. Far-away points usually belong to different topics.

Think of the campus assistant as learning neighborhoods:

- tuition and billing questions cluster together
- parking and vehicle questions cluster together
- dorm repair questions cluster together
- library access questions cluster together
- advising and registration questions cluster together

The classifier does not need to memorize every possible phrase. It learns which neighborhood a new question lands in.

In [ ]:
# Environment setup
import sys
import subprocess
from typing import Any, Dict, List

required_packages = [
    "sentence-transformers",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "plotly",
    "torch",
    "ipywidgets",
]

for package in required_packages:
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import torch
import ipywidgets as widgets
from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

np.random.seed(42)
torch.manual_seed(42)

print("Environment ready.")

## 2. Campus intent dataset

We will use the Phase 2 CSV dataset when it is available. It contains a broader set of realistic student questions across five campus service categories. Each category should become a small neighborhood in the embedding space.

The inline examples below are only a fallback for standalone notebook use.

In [ ]:
from pathlib import Path

dataset_path = Path("data/phase2_intent_queries.csv")

if dataset_path.exists():
    df_dataset = pd.read_csv(dataset_path)
else:
    RAW_CAMPUS_INTENT_DATA = [
        ("How do I pay my tuition for the upcoming fall semester?", "tuition_payment"),
        ("What is the deadline for bursar fee deposits?", "tuition_payment"),
        ("Where do I send an international wire transfer for my college bill?", "tuition_payment"),
        ("Can I set up a monthly installment plan for tuition?", "tuition_payment"),
        ("Are there late payment fees if I pay after the third week?", "tuition_payment"),
        ("Where can I park my car on campus?", "parking_permit"),
        ("How do I register for a commuter vehicle permit?", "parking_permit"),
        ("What is the cost of a campus parking decal?", "parking_permit"),
        ("Can I appeal a parking ticket?", "parking_permit"),
        ("How do I update my license plate on my permit?", "parking_permit"),
        ("The heater in my dorm room is blowing cold air.", "dorm_maintenance"),
        ("How do I submit a plumbing work order for a clogged sink?", "dorm_maintenance"),
        ("My dorm room keycard will not unlock.", "dorm_maintenance"),
        ("There is water leaking from the ceiling in the bathroom.", "dorm_maintenance"),
        ("Who fixes a broken dorm window lock?", "dorm_maintenance"),
        ("What time does the main campus library close tonight?", "library_hours"),
        ("How do I reserve a study room in the library?", "library_hours"),
        ("Are the library archives open on Sunday mornings?", "library_hours"),
        ("How do I check out a book for more than two weeks?", "library_hours"),
        ("Can I book a quiet room for group study?", "library_hours"),
        ("How do I make an appointment with my academic advisor?", "academic_advising"),
        ("What is the deadline to drop a course without penalty?", "academic_advising"),
        ("How do I declare or change my major?", "academic_advising"),
        ("How do I request a degree audit?", "academic_advising"),
        ("What is the procedure for a leave of absence?", "academic_advising"),
    ]
    df_dataset = pd.DataFrame(RAW_CAMPUS_INTENT_DATA, columns=["query_text", "intent_label"])

df_dataset = df_dataset.dropna(subset=["query_text", "intent_label"]).reset_index(drop=True)

display(df_dataset)
print(f"Loaded {len(df_dataset)} training examples.")
print(df_dataset["intent_label"].value_counts())

## 3. Inspect the 384-dimensional vectors

The model `all-MiniLM-L6-v2` turns each question into 384 numbers. Those numbers are coordinates. A single coordinate does not translate cleanly into a word like `parking` or `tuition`; the full pattern is what matters.

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
queries = df_dataset["query_text"].tolist()
X_embeddings = encoder.encode(queries, normalize_embeddings=True, show_progress_bar=True)

label_names = sorted(df_dataset["intent_label"].unique())
label2id = {label: index for index, label in enumerate(label_names)}
id2label = {index: label for label, index in label2id.items()}
y_labels = df_dataset["intent_label"].map(label2id).to_numpy()

print(f"Embedding matrix shape: {X_embeddings.shape}")
print(f"First 12 coordinates: {np.round(X_embeddings[0][:12], 4).tolist()}")
print(f"Label mapping: {label2id}")

## 4. Distance sandbox

Type your own sentence pairs into the boxes below. The cosine similarity and Euclidean distance update instantly, so you can feel how paraphrases stay close together while unrelated topics drift apart.


In [ ]:
def compare_queries(query_a: str, query_b: str) -> None:
    vec_a = encoder.encode([query_a], normalize_embeddings=True)[0]
    vec_b = encoder.encode([query_b], normalize_embeddings=True)[0]
    cosine_sim = float(np.dot(vec_a, vec_b))
    euclidean_dist = float(np.linalg.norm(vec_a - vec_b))
    print(f"Cosine similarity: {cosine_sim:.4f}")
    print(f"Euclidean distance: {euclidean_dist:.4f}")

widgets.interact(
    compare_queries,
    query_a=widgets.Text(
        value="Where can I park my car on campus?",
        description="Query A:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    ),
    query_b=widgets.Text(
        value="How do I get a commuter vehicle permit?",
        description="Query B:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    ),
);


## 5. Fill-in-the-blank training

The next cell is intentionally incomplete. Complete the classifier creation, fit call, and prediction call. Use the scikit-learn `LogisticRegression` documentation as your guide.

Suggested settings: `multi_class="multinomial"`, `max_iter=1000`, `C=1.0`, `random_state=42`.

Once you finish the blanks, run the `test_classifier()` self-check that follows for instant verification, just like the Phase 1 preprocessing checks.


In [ ]:
X_train, X_test, y_train, y_test, queries_train, queries_test = train_test_split(
    X_embeddings,
    y_labels,
    queries,
    test_size=0.20,
    random_state=42,
    stratify=y_labels,
)

# 1. CREATE THE CLASSIFIER
# Hint: set multi_class="multinomial" and max_iter=1000
clf = LogisticRegression(
    multi_class=...,  # TODO: Replace '...' with "multinomial"
    max_iter=...,  # TODO: Replace '...' with 1000
    C=1.0,
    random_state=42,
)

# 2. TRAIN THE CLASSIFIER
# Hint: use clf.fit(X_train, y_train)
...  # TODO: Replace '...' with clf.fit(X_train, y_train)

# 3. PREDICT ON THE TEST SET
# Hint: use clf.predict(X_test)
y_pred = ...  # TODO: Replace '...' with clf.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=label_names))


In [ ]:
def test_classifier():
    assert "clf" in globals(), "Define clf before running this test."
    params = clf.get_params()
    assert params["multi_class"] == "multinomial", "Set multi_class='multinomial' on the classifier."
    assert params["max_iter"] == 1000, "Set max_iter=1000 on the classifier."
    test_accuracy = accuracy_score(y_test, clf.predict(X_test))
    assert test_accuracy >= 0.70, f"Accuracy too low ({test_accuracy:.2%}). Check your training setup."
    print("  ✓ Classifier configuration verified (multi_class='multinomial', max_iter=1000).")
    print(f"  ✓ Test accuracy {test_accuracy:.2%} meets the required threshold.")
    print("🎉 All classifier self-checks passed!")

test_classifier()


In [ ]:
def predict_intent(query: str, encoder: SentenceTransformer, classifier: Any) -> Dict[str, Any]:
    embedding = encoder.encode([query], normalize_embeddings=True)[0]
    probabilities = classifier.predict_proba([embedding])[0]
    predicted_index = int(np.argmax(probabilities))
    return {
        "query": query,
        "predicted_intent": id2label[predicted_index],
        "confidence_score": round(float(probabilities[predicted_index]), 4),
        "class_distribution": {id2label[i]: round(float(prob), 4) for i, prob in enumerate(probabilities)},
        "embedding": embedding,
    }

print("Prediction helper ready. It will work after students define and fit clf.")

### Break the engine challenge

The classifier only knows the neighborhoods it was trained on. Try to write a deceptive query that blends vocabulary from two different intents, for example mixing "dorm maintenance" words with "tuition" words, and watch the probability distribution below get confused.


In [ ]:
adversarial_query_box = widgets.Text(
    value="I need a work order to fix the tuition bill heater in my dorm room.",
    description="Tricky query:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

def break_the_engine(query: str) -> None:
    result = predict_intent(query, encoder, clf)
    print(f"Predicted intent: {result['predicted_intent']}")
    print(f"Confidence: {result['confidence_score']:.4f}")
    print("Class distribution:")
    for label, prob in sorted(result["class_distribution"].items(), key=lambda item: item[1], reverse=True):
        print(f"  {label}: {prob:.4f}")

widgets.interact(break_the_engine, query=adversarial_query_box);


## 6. Visualize the semantic neighborhoods

We cannot draw 384 dimensions directly, so we project the embeddings into 2D and 3D with PCA. This is like turning a detailed city map into a simplified teaching map: some detail is lost, but the main neighborhoods should still be visible.

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_embeddings)

df_2d = pd.DataFrame({
    "pc1": X_2d[:, 0],
    "pc2": X_2d[:, 1],
    "intent": [id2label[label] for label in y_labels],
    "query": queries,
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_2d, x="pc1", y="pc2", hue="intent", s=90, palette="tab10")
plt.title("2D PCA view of campus intent neighborhoods")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

pca_3d = PCA(n_components=3, random_state=42)
X_3d = pca_3d.fit_transform(X_embeddings)

df_3d = pd.DataFrame({
    "pc1": X_3d[:, 0],
    "pc2": X_3d[:, 1],
    "pc3": X_3d[:, 2],
    "intent": [id2label[label] for label in y_labels],
    "query": queries,
})

fig = px.scatter_3d(
    df_3d,
    x="pc1",
    y="pc2",
    z="pc3",
    color="intent",
    hover_name="query",
    title="Interactive 3D semantic neighborhood map",
)
fig.update_traces(marker=dict(size=5))
fig.show()

## 7. OOD rejection with a JSON-style fallback

A classifier trained on five campus intents is a closed-world system. It will try to route every question into one of those known neighborhoods, even if the question is not about campus services.

A confidence threshold gives the assistant permission to say, in effect: `This does not look like one of my known categories.` Complete the "Confidence Bouncer" logic below, mirroring the fallback mechanism you built in Phase 1.


In [ ]:
def predict_intent_with_fallback(
    query: str,
    encoder: SentenceTransformer,
    classifier: Any,
    confidence_threshold: float = 0.50,
) -> Dict[str, Any]:
    result = predict_intent(query, encoder, classifier)
    score = result["confidence_score"]

    # Hint: Check if score is LESS THAN confidence_threshold
    if ...:  # TODO: Replace '...' with the correct math comparison
        return {
            "status": "...",  # TODO: Replace '...' with "OOD_FALLBACK_TRIGGERED"
            "query": query,
            "confidence_score": score,
            "fallback_message": "This question appears to be outside the known campus support categories.",
        }

    return {
        "status": "...",  # TODO: Replace '...' with "CONFIRMED_INTENT"
        "query": query,
        "predicted_intent": result["predicted_intent"],
        "confidence_score": score,
        "class_distribution": result["class_distribution"],
    }

# -----------------------------------------------------------------------------
# TEST THE BOUNCER
# -----------------------------------------------------------------------------
for test_query in [
    "Where can I park my car on campus?",
    "What is the best pizza place near the football stadium?",
]:
    response = predict_intent_with_fallback(test_query, encoder, clf)
    print(f"Query: '{test_query}'")
    print(f"Status: {response['status']}")
    print(f"Confidence: {response['confidence_score']:.4f}\n")


In [ ]:
custom_intent_examples = [
    ("How do I get a replacement student ID card?", "student_id_services"),
    ("I lost my campus ID and need a reprint.", "student_id_services"),
    ("Where do I update my student photo for ID access?", "student_id_services"),
    ("My student ID is not working at the library entrance.", "student_id_services"),
    ("How do I activate my new campus card?", "student_id_services"),
    ("Can I get a temporary digital ID for access?", "student_id_services"),
    ("Where do I request a student card replacement after damage?", "student_id_services"),
    ("My card was stolen; how do I deactivate it?", "student_id_services"),
    ("What is the fee for a replacement student ID?", "student_id_services"),
    ("How do I update my student identification record?", "student_id_services"),
]

print("Custom intent challenge examples ready. Add these to the dataset, rebuild labels, and retrain.")

## Summary

Dense embeddings let us treat questions as points in a learned meaning map. The classifier learns the service neighborhoods, the distance sandbox makes semantic closeness visible, and the fallback function prepares the assistant for questions that do not belong to the known campus routes.

This creates the bridge to Phase 3: using retrieval-augmented generation to turn a good route into a grounded answer.